# 添付ノートブックへの追加
元のPKCE生成・Drive接続セルを残しています。PKCE生成は今回のエージェント実行には不要で、スキップできます。認証情報を出力・共有しないでください。
以下にローカル検証と提出ZIP生成、任意の公式ハーネス評価を追加しました。元ファイルの空セルに残っていた過去のSyntaxError出力は引き継いでいません。


In [ ]:
import secrets
import hashlib
import base64

# Generate a random code_verifier (43-128 characters)
code_verifier = secrets.token_urlsafe(32)

# Create code_challenge using SHA-256
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).decode().rstrip('=')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Gemma 4 エージェント：Colab で検証・評価

まず `dist/colab-source.zip` をアップロードしてテストと提出ZIP生成を実行します。未コミットの実装も含むため、GitHub に未反映でも使えます。

推論評価は任意の後半セルで実行します。コンペ提供の公式ハーネスとその依存関係、ローカル推論モデル、公開タスク、スナップショットが別途必要です。重みのアクセス条件・コンペ規約は利用者が確認してください。このノートブックに認証情報を記入しないでください。

Colab のGPU割当は公式 L4×4 と異なります。量子化モデルでも、ハーネスの推論バックエンド・並列数がGPU構成に対応する必要があります。GPUを検出しただけでは推論可能とは判定しません。

In [ ]:
from google.colab import files
from pathlib import Path
import stat
import zipfile

uploaded = files.upload()  # colab-source.zip を選択
if len(uploaded) != 1:
    raise ValueError("colab-source.zip を1つ選択してください")
archive_name = next(iter(uploaded))
project = Path('/content/gemma-agent')
if project.exists():
    raise FileExistsError('既存ファイル保護のため停止します。別の展開先を指定してください。')
with zipfile.ZipFile(archive_name) as archive:
    for info in archive.infolist():
        target = (project / info.filename).resolve()
        if not target.is_relative_to(project.resolve()) or stat.S_ISLNK(info.external_attr >> 16):
            raise ValueError(f'安全でないZIPエントリー: {info.filename}')
    archive.extractall(project)
print(project)


In [ ]:
import subprocess
import sys

def run(args, cwd=project):
    subprocess.run(args, cwd=cwd, check=True)

run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-dev.txt'])
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])
run([sys.executable, 'tools/package_submission.py'])
files.download(str(project / 'dist/submission.zip'))


## 任意：公式ハーネスでの推論評価

以下はローカル評価用です。上の提出ZIP生成には不要です。

1. 必要なら次のセルで Google Drive を接続します。
2. コンペ配布物の手順で `swegemma` と依存関係をこのランタイムに導入してください。出所未確認の同名PyPIパッケージは自動インストールしません。
3. モデル重み、`tasks.jsonl`、スナップショット、ハーネス形式の `models.yaml` を用意します。`models.yaml` のモデル別設定は配布例に従い、このランタイムのローカル推論サーバーに接続してください。ノートブックはサーバーを自動起動しません。
4. GPU数と推論バックエンドの対応を確認します。公式 `scripts/inference.py` の4GPU設定を単一GPUにそのまま使用しないでください。

Dockerデーモンを通常使えないColabでは `subprocess` を指定します。これはDocker隔離と異なり、タスクのコマンドをランタイム上で実行します。信頼できる公開タスクを使用し、評価用のランタイムに秘密情報や不要なDriveファイルを置かないでください。

In [ ]:
import shutil

if shutil.which('nvidia-smi'):
    run(['nvidia-smi'], cwd='/content')
else:
    print('GPU未検出。提出ZIPの生成は可能ですが、このノートブックではモデル推論を検証できません。')
if not shutil.which('swegemma'):
    raise RuntimeError('公式配布手順で swegemma を導入してから再実行してください。')
run(['swegemma', 'eval', '--help'])


In [ ]:
# 実際の配布物の場所に変更してください。
TASKS = Path('/content/data/tasks.jsonl')
SNAPSHOTS = Path('/content/data/snapshots')
MODELS_YAML = Path('/content/data/models.yaml')
TASK_ID = ''  # 最初は公開タスクの instance_id を1つ指定

for path in (TASKS, SNAPSHOTS, MODELS_YAML):
    if not path.exists():
        raise FileNotFoundError(path)
if not TASK_ID.strip():
    raise ValueError('最初の動作確認用に TASK_ID を指定してください。')

from datetime import datetime, timezone
results = project / 'results' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
command = [
    'swegemma', 'eval',
    '--tasks', str(TASKS), '--snapshots-dir', str(SNAPSHOTS),
    '--submission-dir', str(project / 'submission'),
    '--models-yaml', str(MODELS_YAML),
    '--results-dir', str(results), '--task-id', TASK_ID,
    '--sandbox', 'subprocess', '--concurrency', '1',
    '--max-tool-calls', '50', '--max-time-minutes', '30',
    '--display', 'auto',
]
run(command)
summary = results / 'summary.json'
if not summary.is_file():
    raise RuntimeError('評価サマリーが生成されていません。ログを確認してください。')
print(summary.read_text())


`summary.json`、`task_results.jsonl`、`test_outputs/`、`patches/` を確認してください。コマンドの正常終了だけで issue 解決とは判定せず、必要なテストが実行されて成功しているかを確認します。Colabでの結果は公式Kaggle評価の代替ではありません。LoRA学習はこのノートブックには含みません。